# DS2002 · SQL to Python to DataFrames

**Lecture — 2026-09-14 · Fall 2026**  
**Class time:** 45 minutes

---

## The same operations, a different tool

You spent last week describing results in SQL. Today you do the same work in Pandas, and the honest question is why bother learning both.

Use **SQL** when the data already lives in a database, when it is too big to fit in memory, or when someone else needs to run your query. Use **Pandas** when the data arrives as files, when you need to reshape it repeatedly while you figure out what you are looking at, or when the next step is a chart or a model.

Most real pipelines use both: SQL to pull a manageable slice, Pandas to work it. Here is the translation table — pin it somewhere.

| SQL | Pandas |
|---|---|
| `SELECT a, b FROM t` | `df[['a', 'b']]` |
| `WHERE qty > 2` | `df[df['qty'] > 2]` |
| `WHERE a > 2 AND b = 'x'` | `df[(df['a'] > 2) & (df['b'] == 'x')]` |
| `ORDER BY qty DESC` | `df.sort_values('qty', ascending=False)` |
| `GROUP BY cat` + `SUM(x)` | `df.groupby('cat')['x'].sum()` |
| `HAVING SUM(x) > 100` | `.loc[lambda s: s > 100]` after the groupby |
| `JOIN ... ON a.id = b.id` | `a.merge(b, on='id')` |
| `COUNT(DISTINCT u)` | `df['u'].nunique()` |
| `LIMIT 5` | `df.head(5)` |

### The data

Two files, the way you would actually get them: order lines from the point-of-sale system, and a vendor list from somebody's spreadsheet.

In [ ]:
import pandas as pd
from io import StringIO

orders = pd.read_csv(StringIO('''order_id,vendor_id,category,item,qty,price
1001,V-01,Food,Cheeseburger,3,7.50
1002,V-10,Merch,Foam Finger,1,12.00
1003,V-10,Merch,UVA T-Shirt,2,24.00
1004,V-05,Food,Chicken Tacos,4,6.50
1005,V-18,RainGear,Rain Poncho,10,6.00
1006,V-01,Food,Hot Dog,2,4.50
1007,V-99,Drink,Bottled Water,6,3.00'''))

vendors = pd.read_csv(StringIO('''vendor_id,vendor_name,zone
V-01,Hoos Burgers,A
V-05,Rotunda Tacos,B
V-10,Cav Merch North,A
V-18,Rally Rain Gear,C'''))

print(orders.shape, vendors.shape)
orders

### Look before you work

`.info()` is the Pandas equivalent of reading the schema. It tells you the row count, the column types, and how many non-null values each column has — three things that will explain most errors you are about to hit.

In [ ]:
orders.info()

### Selecting and filtering

One bracket gives you a Series; two brackets give you a DataFrame. That distinction matters as soon as you start chaining operations.

In [ ]:
print(type(orders['item']))          # Series -- one column
print(type(orders[['item', 'qty']])) # DataFrame -- a table

orders[orders['category'] == 'Food'][['item', 'qty', 'price']]

Combining conditions in Pandas needs `&` and `|`, not `and` and `or`, and every condition needs its own parentheses. Forgetting either produces one of the least helpful error messages in Python.

In [ ]:
orders[(orders['qty'] >= 3) & (orders['price'] < 10)]

### Derived columns and the copy warning

Adding a column is straightforward. What trips people up is adding one to a **slice**.

In [ ]:
orders['revenue'] = orders['qty'] * orders['price']
orders[['order_id', 'item', 'qty', 'price', 'revenue']]

Now the mistake. This looks reasonable and pandas will warn you about it:

```python
food = orders[orders['category'] == 'Food']
food['discounted'] = food['revenue'] * 0.9      # SettingWithCopyWarning
```

The slice may be a view onto the original, so pandas cannot promise where your assignment lands. The fix is one word — `.copy()` — and it should become reflex whenever you filter and then modify.

In [ ]:
food = orders[orders['category'] == 'Food'].copy()
food['discounted'] = (food['revenue'] * 0.9).round(2)
food[['item', 'revenue', 'discounted']]

### Grouping

`groupby` splits rows into buckets, applies a function, and puts the answers back together. This is `GROUP BY` with more flexibility about what comes next.

In [ ]:
orders.groupby('category')['revenue'].sum().sort_values(ascending=False)

You will often want several summaries at once. `.agg()` takes a dict of `new_name=(column, function)` pairs, which keeps the result readable.

In [ ]:
orders.groupby('category').agg(
    orders=('order_id', 'count'),
    units=('qty', 'sum'),
    revenue=('revenue', 'sum'),
    avg_ticket=('revenue', 'mean'),
).round(2)

### Joining, and checking that the join was honest

Here is the move Monday's SQL lecture spent all week building toward, in Pandas. `orders` has a `vendor_id`; the names live in `vendors`.

Note `how='left'` and `indicator=True`. The indicator column tells you, row by row, whether the match succeeded — which is how you catch a silent data problem instead of shipping it.

In [ ]:
joined = orders.merge(vendors, on='vendor_id', how='left', indicator=True)
print('rows before:', len(orders), '| rows after:', len(joined))
print()
print(joined['_merge'].value_counts())
joined[['order_id', 'vendor_id', 'vendor_name', 'zone', 'revenue', '_merge']]

Seven rows in, seven rows out, but one of them did not match: vendor `V-99` is in the orders and not in the vendor list. Its `vendor_name` and `zone` are `NaN`.

That is a real finding, not a nuisance. Somebody sold six bottles of water and the vendor roster does not know who they are. If you had used `how='inner'` you would have quietly dropped eighteen dollars and never known.

In [ ]:
unmatched = joined[joined['_merge'] == 'left_only']
print('orders with no known vendor:', len(unmatched))
print('revenue at stake: $', unmatched['revenue'].sum(), sep='')

### The three questions to ask after every merge

1. **Did the row count change the way I expected?** More rows than you started with means your join key is not unique on one side and rows got duplicated.
2. **Did anything fail to match?** `indicator=True`, then look at `_merge`.
3. **Are the totals still right?** Sum a money column before and after. If it moved, the join did something you did not ask for.

You will do exactly this on the midterm when you join sales to weather, and on the capstone when you join orders to vendors. Practicing it on seven rows is much cheaper than debugging it on a hundred thousand.

In [ ]:
# The duplication trap, in miniature: a vendor list with a repeated id
dupes = pd.concat([vendors, vendors.iloc[[0]]], ignore_index=True)
bad = orders.merge(dupes, on='vendor_id', how='left')
print('orders:', len(orders), '-> after joining a list with a duplicate id:', len(bad))
print('revenue before: $', orders['revenue'].sum(), sep='')
print('revenue after:  $', bad['revenue'].sum(), '  <- inflated', sep='')

Nothing errored. The revenue total just went up because one order matched two vendor rows and got counted twice. This is the most common way a Pandas pipeline produces a confident wrong number.

### Practice 1 — units per vendor

Total quantity sold per vendor **name** (not id), highest first. You will need the joined frame.

In [ ]:
# TODO

### Practice 2 — revenue by zone

Revenue per zone. Decide what to do with the order whose vendor is unknown, and say what you decided in a comment.

In [ ]:
# TODO

### Practice 3 — the SQL translation

Write the Pandas equivalent of:

```sql
SELECT category, SUM(revenue) AS revenue
FROM orders
WHERE qty >= 2
GROUP BY category
HAVING SUM(revenue) > 20
ORDER BY revenue DESC
```

In [ ]:
# TODO